## 🎯 Learning Objectives
* Understand the concept of fan-out and fan-in patterns in agentic workflows.
* Learn how to design and implement parallel execution paths in LangGraph using state management and graph topology.
* Identify appropriate use cases for fan-out/fan-in to improve efficiency and throughput in complex AI agents.
* Analyze the performance implications and trade-offs of parallel agent execution.


## Parallelism with Fan-Out and Fan-In Nodes in LangGraph

In the realm of complex AI agents, efficiency and responsiveness are paramount. Often, a single high-level task can be decomposed into multiple independent sub-tasks that can be executed concurrently. This is where the **fan-out** and **fan-in** patterns become incredibly powerful.

### What are Fan-Out and Fan-In?

Imagine you're a project manager (your main agent) tasked with creating a comprehensive report. To speed things up, you don't do everything yourself. Instead:

1.  **Fan-Out (Delegation):** You delegate different parts of the report to specialized teams simultaneously. For example, one team researches market trends, another drafts the executive summary, and a third compiles financial data. This is the **fan-out** phase – a single input (the report request) leads to multiple parallel processing paths.

2.  **Parallel Execution:** While these teams work independently, they don't wait for each other. They leverage their specialized skills to complete their assigned tasks concurrently.

3.  **Fan-In (Aggregation):** Once all teams have completed their parts, you (the project manager) gather all their individual contributions. You then synthesize, combine, and refine these pieces into the final, cohesive report. This is the **fan-in** phase – multiple parallel results converge back into a single, unified output.

### Why is this important for Advanced AI Agents?

*   **Improved Throughput & Latency:** By executing independent tasks in parallel, the overall time to complete a complex workflow can be significantly reduced. This is crucial for real-time applications or scenarios requiring rapid responses.
*   **Modularity & Specialization:** Each parallel branch can host a specialized agent or tool, allowing for a clear separation of concerns and leveraging the best-suited models or techniques for each sub-task.
*   **Scalability:** As the complexity of the main task grows, you can easily add more parallel branches or scale resources for existing ones without redesigning the entire workflow.
*   **Resilience:** If one parallel branch encounters an issue, others can continue, and the system can potentially recover or adapt, though careful error handling is required.

### How LangGraph Facilitates Fan-Out and Fan-In

LangGraph, with its state-driven execution model, is exceptionally well-suited for implementing these patterns:

1.  **State Management:** The shared `AgentState` is key. A 


In [ ]:
import asyncio
from typing import TypedDict, List, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda

# --- 2026 Ready Mock LLM for demonstration ---
# In a real 2026 scenario, you'd use advanced models like GPT-5, Gemini Ultra, Claude 4, etc.
# For this example, we'll use a simple mock to ensure reproducibility and speed.
class MockLLM:
    def __init__(self, name: str, delay: float = 0.5):
        self.name = name
        self.delay = delay

    async def invoke(self, prompt: str) -> str:
        await asyncio.sleep(self.delay) # Simulate network latency or computation
        print(f"[{self.name}]: Processing prompt: '{prompt[:50]}...' ")
        if "research" in self.name.lower():
            return f"Detailed research findings for '{prompt}'. Key points: A, B, C. (from {self.name})"
        elif "draft" in self.name.lower():
            return f"Initial draft outline for '{prompt}'. Sections: Intro, Body, Conclusion. (from {self.name})"
        elif "summarize" in self.name.lower():
            return f"Comprehensive report based on: {prompt}. (from {self.name})"
        return f"Processed: {prompt} (from {self.name})"

# --- Define the Agent State ---
# This TypedDict defines the schema for our graph's state.
# Each node will receive and return an instance of this state, updating relevant fields.
class AgentState(TypedDict):
    query: str
    research_result: str
    draft_result: str
    final_report: str
    # We can add a history for debugging or conversational context
    messages: List[BaseMessage]

# --- Define the Nodes (Agents/Functions) ---

# 1. Planner Node (Fan-Out Initiator)
# This node takes the initial query and prepares the state for parallel tasks.
async def planner_node(state: AgentState) -> AgentState:
    print("\n--- PLANNER NODE: Initiating parallel tasks ---")
    query = state["query"]
    # In a real scenario, the planner might break down the query into sub-queries
    # or decide which parallel agents to activate based on the query content.
    print(f"Planner received query: '{query}'")
    # Initialize results to None or empty strings to indicate they are pending
    return {**state, "research_result": None, "draft_result": None}

# 2. Research Agent Node (Parallel Task 1)
# This agent performs research based on the query.
research_llm = MockLLM("ResearchAgentLLM", delay=1.0)
async def research_agent_node(state: AgentState) -> AgentState:
    print("\n--- RESEARCH AGENT: Performing research ---")
    query = state["query"]
    research_output = await research_llm.invoke(f"Perform in-depth research on: {query}")
    print(f"Research Agent completed. Output snippet: '{research_output[:70]}...' ")
    return {**state, "research_result": research_output}

# 3. Drafting Agent Node (Parallel Task 2)
# This agent drafts content based on the query.
drafting_llm = MockLLM("DraftingAgentLLM", delay=0.8)
async def drafting_agent_node(state: AgentState) -> AgentState:
    print("\n--- DRAFTING AGENT: Drafting content ---")
    query = state["query"]
    draft_output = await drafting_llm.invoke(f"Create an outline and initial draft for: {query}")
    print(f"Drafting Agent completed. Output snippet: '{draft_output[:70]}...' ")
    return {**state, "draft_result": draft_output}

# 4. Aggregator Node (Fan-In & Final Synthesis)
# This node waits for both parallel tasks to complete and then combines their results.
aggregator_llm = MockLLM("AggregatorLLM", delay=0.7)
async def aggregator_node(state: AgentState) -> AgentState:
    print("\n--- AGGREGATOR NODE: Combining results ---")
    query = state["query"]
    research_result = state["research_result"]
    draft_result = state["draft_result"]

    # Ensure both results are available before proceeding
    if research_result is None or draft_result is None:
        raise ValueError("Aggregator received incomplete results from parallel tasks.")

    combined_input = (
        f"Original Query: {query}\n\n"
        f"Research Findings:\n{research_result}\n\n"
        f"Draft Outline:\n{draft_result}"
    )
    final_report = await aggregator_llm.invoke(f"Synthesize the following information into a comprehensive report: {combined_input}")
    print(f"Aggregator completed. Final report snippet: '{final_report[:70]}...' ")
    return {**state, "final_report": final_report}

# --- Build the LangGraph Workflow ---

workflow = StateGraph(AgentState)

# Add the nodes to the graph
workflow.add_node("planner", planner_node)
workflow.add_node("research_agent", research_agent_node)
workflow.add_node("drafting_agent", drafting_agent_node)
workflow.add_node("aggregator", aggregator_node)

# Set the entry point for the graph
workflow.set_entry_point("planner")

# Define the fan-out transitions:
# From the 'planner', we want both 'research_agent' and 'drafting_agent' to be eligible to run.
# In LangGraph, if multiple nodes have their dependencies met (i.e., the state they need is available),
# and there are edges leading to them, they can execute concurrently, especially with an async executor.
# We explicitly add edges from 'planner' to both parallel agents.
workflow.add_edge("planner", "research_agent")
workflow.add_edge("planner", "drafting_agent")

# Define the fan-in transitions:
# Both 'research_agent' and 'drafting_agent' should transition to the 'aggregator'.
# LangGraph's default behavior for multiple incoming edges to a node is to wait
# for all upstream nodes to complete and update the state before executing the target node.
# This naturally creates the fan-in (join) point.
workflow.add_edge("research_agent", "aggregator")
workflow.add_edge("drafting_agent", "aggregator")

# From the 'aggregator', the workflow ends.
workflow.add_edge("aggregator", END)

# Compile the graph
# Using AsyncLocalGraphExecutor allows for true concurrent execution of nodes
# that are ready to run, which is ideal for demonstrating parallelism.
app = workflow.compile()

# --- Run the Graph ---

# Define the initial input for our agent workflow
initial_input = {"query": "The impact of quantum computing on cybersecurity by 2030"}

print(f"\n--- Starting Agent Workflow for query: '{initial_input['query']}' ---")

# Invoke the compiled graph asynchronously
# The `stream()` method is useful for observing intermediate steps, but `invoke()`
# will give us the final state directly.
final_state = asyncio.run(app.ainvoke(initial_input))

print("\n--- WORKFLOW COMPLETED ---")
print("\nFinal Report:")
print(final_state["final_report"])

# You can also inspect other parts of the final state
# print("\nResearch Result:")
# print(final_state["research_result"])
# print("\nDraft Result:")
# print(final_state["draft_result"])


### Interpreting the Code Output and Performance Trade-offs

When you run the provided code, you'll observe the following:

1.  **Sequential Start:** The `PLANNER NODE` executes first, as it's the entry point.
2.  **Concurrent Execution:** Immediately after the planner, you'll see print statements indicating that both the `RESEARCH AGENT` and `DRAFTING AGENT` begin their tasks. Notice that their `Processing prompt` messages might appear interleaved, and their `completed` messages will likely appear out of order, reflecting their independent, concurrent execution. The `delay` values in our `MockLLM`s simulate different processing times, further highlighting that they don't wait for each other.
3.  **Fan-In Synchronization:** The `AGGREGATOR NODE` will *only* start executing after *both* the `RESEARCH AGENT` and `DRAFTING AGENT` have completed their tasks and updated the shared state. LangGraph's state-driven execution naturally handles this synchronization: a node won't run until all its upstream dependencies (nodes whose outputs it relies on, or nodes that feed into its state) have finished.
4.  **Final Output:** The aggregator then synthesizes the results into the `Final Report`.

#### Performance Trade-offs:

**Advantages:**

*   **Reduced Latency:** The most significant benefit. If `research_agent` takes 10 seconds and `drafting_agent` takes 8 seconds, running them sequentially would take 18 seconds. In parallel, the total time is closer to the maximum of the two (10 seconds), plus a small overhead for orchestration.
*   **Increased Throughput:** More tasks can be processed in a given time frame, especially if the parallel tasks are I/O-bound (e.g., making API calls to different services).
*   **Better Resource Utilization:** If your system has multiple CPU cores or can handle concurrent I/O, parallelism ensures these resources are actively used.

**Disadvantages:**

*   **Increased Complexity:** Designing, debugging, and monitoring parallel workflows can be more challenging than sequential ones. State management across concurrent branches requires careful thought to avoid race conditions or unexpected behavior (though LangGraph's immutable state updates help mitigate this).
*   **Overhead:** There's a small overhead associated with managing parallel execution (e.g., context switching, task scheduling). For very short, CPU-bound tasks, this overhead might negate the benefits.
*   **Resource Contention:** If parallel tasks compete for the same limited resources (e.g., a single GPU, a rate-limited API), actual performance gains might be less than theoretical, or even lead to bottlenecks.
*   **Debugging:** Tracing the flow of data and control in a parallel system can be harder, especially when trying to pinpoint the source of an error.

### Typical Use Cases:

*   **Multi-Modal Processing:** Simultaneously analyzing text, images, and audio inputs for a comprehensive understanding.
*   **Retrieval Augmented Generation (RAG):** Running multiple retrieval strategies (e.g., semantic search, keyword search, graph database lookup) in parallel, then aggregating results for the LLM.
*   **Expert Agent Collaboration:** Sending a query to multiple specialized agents (e.g., a legal expert, a medical expert, a financial analyst) and then having a supervisor agent synthesize their independent assessments.
*   **Data Preprocessing Pipelines:** Performing different cleaning, transformation, or feature engineering steps on data concurrently.
*   **A/B Testing / Parallel Evaluation:** Running multiple versions of a sub-agent or different strategies in parallel to compare their outputs or performance.
*   **Complex Decision Making:** Gathering information from various sources or performing different types of analysis (e.g., sentiment analysis, entity extraction, fact-checking) before making a final decision.


### Resources

*   **LangGraph Documentation:**
    *   [LangGraph Introduction](https://langchain-ai.github.io/langgraph/)
    *   [LangGraph StateGraph Tutorial](https://langchain-ai.github.io/langgraph/tutorials/introduction/)
    *   [LangGraph Executors (for understanding async execution)](https://langchain-ai.github.io/langgraph/reference/executors/)
*   **LangChain Expression Language (LCEL):**
    *   [LCEL `RunnableParallel` (for parallelism within a single runnable)](https://python.langchain.com/docs/expression_language/how_to/map)
*   **Asynchronous Python:**
    *   [Python `asyncio` Documentation](https://docs.python.org/3/library/asyncio.html)
*   **Advanced LangGraph Patterns:**
    *   Look for official LangChain/LangGraph blog posts or examples on complex graph topologies, which often feature parallel execution for real-world applications.
